In [ ]:
import pandas as pd, snowflake.connector

SNOWFLAKE_CONFIG = {
    'user':     "",
    "password": "",
    "account": "",
    "warehouse": "TAXI_WH",
    "database": "TAXI_DB",
    "schema": "RAW"
}

conn = snowflake.connector.connect(**SNOWFLAKE_CONFIG)
df   = pd.read_sql('SELECT * FROM TAXI_DB.MARTS.MART_TRIP_FEATURES', conn)
df.columns = df.columns.str.lower()
conn.close()


print(f'Shape: {df.shape}')
print(f'Tip rate: {df["tipped"].mean():.2%}')

C:\Users\brand\AppData\Local\Temp\ipykernel_14604\1052852641.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df   = pd.read_sql('SELECT * FROM TAXI_DB.MARTS.MART_TRIP_FEATURES', conn)


Shape: (130608, 15)
Tip rate: 88.59%


In [2]:
import mlflow, mlflow.sklearn
from sklearn.ensemble        import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics         import (roc_auc_score, average_precision_score,
                                     brier_score_loss, classification_report)
from mlflow.models.signature import infer_signature


FEATURES = ['pickup_hour','pickup_dow','pickup_month','is_weekend','is_late_night',
            'trip_miles','trip_minutes','fare_per_mile','fare_per_minute',
            'extras','fare','pickup_is_airport','dropoff_is_airport']
TARGET   = 'tipped'


X = df[FEATURES].fillna(df[FEATURES].median())
y = df[TARGET]
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)


mlflow.set_tracking_uri('file:./mlruns')
mlflow.set_experiment('chicago_tip_prediction')


params = { 'n_estimators':300, 'learning_rate':0.05, 'max_depth':4,
           'subsample':0.8, 'min_samples_leaf':50, 'random_state':42 }


with mlflow.start_run(run_name='gbt_baseline') as run:
    mlflow.log_params(params)
    mlflow.log_param('dbt_mart', 'TAXI_DB.MARTS.MART_TRIP_FEATURES')


    model  = GradientBoostingClassifier(**params)
    model.fit(X_train, y_train)


    y_prob = model.predict_proba(X_test)[:,1]
    y_pred = model.predict(X_test)


    mlflow.log_metric('roc_auc',       roc_auc_score(y_test, y_prob))
    mlflow.log_metric('avg_precision',  average_precision_score(y_test, y_prob))
    mlflow.log_metric('brier_score',    brier_score_loss(y_test, y_prob))


    print(f'ROC-AUC:       {roc_auc_score(y_test, y_prob):.4f}')
    print(f'Avg Precision: {average_precision_score(y_test, y_prob):.4f}')
    print(f'Brier Score:   {brier_score_loss(y_test, y_prob):.4f}  (lower=better calibration)')
    print(classification_report(y_test, y_pred, target_names=['no tip','tipped']))


    sig = infer_signature(X_train, model.predict_proba(X_train))
    mlflow.sklearn.log_model(model, 'tip_classifier', signature=sig,
                             registered_model_name='chicago_tip_classifier')
    RUN_ID = run.info.run_id
    print(f'Run ID: {RUN_ID}')


c:\Users\brand\portfolio\Tip-Classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\brand\portfolio\Tip-Classification\.venv\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


ROC-AUC:       0.8194
Avg Precision: 0.9611
Brier Score:   0.0590  (lower=better calibration)
              precision    recall  f1-score   support

      no tip       0.93      0.46      0.61      2980
      tipped       0.93      1.00      0.96     23142

    accuracy                           0.93     26122
   macro avg       0.93      0.73      0.79     26122
weighted avg       0.93      0.93      0.92     26122



c:\Users\brand\portfolio\Tip-Classification\.venv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/04/10 16:55:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/10 16:55:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requir

Run ID: dc42c5e9bb4045ba865e7ea606441614


c:\Users\brand\portfolio\Tip-Classification\.venv\Lib\site-packages\mlflow\tracking\_model_registry\utils.py:220: FutureWarning: The filesystem model registry backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri)
Registered model 'chicago_tip_classifier' already exists. Creating a new version of this model...
Created version '2' of model 'chicago_tip_classifier'.
